This notebook cleans the charted data and checks it against the [codebook](../codebook.qmd). The charting form has two tables. `data/raw/included_sources_raw.csv` has one row per included source. `data/raw/included_factors_raw.csv` has one row per factor that a source reports. The notebook writes `data/processed/included_sources_clean.csv`, `data/processed/included_factors_clean.csv`, and `data/processed/bibliography.csv`, which holds the reference details of each included source, taken from the deduplicated records.

The variables, the closed value lists, and the required variables are read from `codebook.yml`, so this notebook holds no copy of them. Cleaning trims whitespace and lowercases the closed variables.

The checks come first. If any check finds an error, the notebook stops and writes nothing. A warning is printed and does not stop the run. Errors are an empty required cell, a value outside the codebook, a repeated `source_id`, a factor whose `source_id` is not in the sources table, a `record_id` that is not in the deduplicated records, and a `source_id` that screening did not include. Warnings are sources with no factors, included sources not charted yet, repeated factor rows, and factors not yet coded to a category or level.

In [1]:
from pathlib import Path

import pandas as pd
import yaml

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "_quarto.yml").exists())
PROC = ROOT / "data" / "processed"
DEDUP = PROC / "deduplicated_records.csv"
INCLUDED = PROC / "included_reports.csv"

TABLES = yaml.safe_load((ROOT / "codebook.yml").read_text(encoding="utf-8"))["tables"]
for t in TABLES.values():
    for v in t["variables"]:
        v["values"] = v.get("values", [])
        if not all(isinstance(x, str) for x in v["values"]):
            raise ValueError(f'codebook.yml: the values of {v["name"]} must be text. Put quotes around values such as "yes", "no", and "1".')


def load(name):
    t = TABLES[name]
    path = ROOT / t["file"]
    cols = [v["name"] for v in t["variables"]]
    if not path.exists():
        return pd.DataFrame(columns=cols)
    df = pd.read_csv(path, dtype=str).fillna("")
    df.columns = [c.strip().lower() for c in df.columns]
    absent = [c for c in cols if c not in df.columns]
    if absent:
        raise ValueError(f"{path.name} is missing column(s): {', '.join(absent)}")
    return df[cols].apply(lambda s: s.str.strip())


def sample(values, n=5):
    values = sorted(set(map(str, values)))
    return ", ".join(values[:n]) + (f" and {len(values) - n} more" if len(values) > n else "")


def lines(mask):
    # Line numbers in the CSV file: the header is line 1.
    numbers = sorted(i + 2 for i in mask[mask].index)
    return ", ".join(map(str, numbers[:5])) + (f" and {len(numbers) - 5} more" if len(numbers) > 5 else "")


sources, factors = load("sources"), load("factors")
have_data = len(sources) > 0 or len(factors) > 0
print(f"{len(sources)} charted source(s), {len(factors)} charted factor(s)." if have_data else "Nothing charted yet. Add rows to the two charting files in data/raw/.")

Nothing charted yet. Add rows to the two charting files in data/raw/.


## OpenAlex field and subfield

This step takes the field and subfield of each included source's primary topic from OpenAlex by DOI, and saves them with the pull date to `data/raw/openalex_topics.csv`. It pulls only sources not pulled before, so values already pulled never change. It then adds a row to `included_sources_raw.csv` for each included source that has none, with the `source_id`, the `record_id` and the OpenAlex values, and `no-topic` in `openalex_corrected` where OpenAlex gives none. Rows already in the file stay as they are.

In [ ]:
import csv
import time
from datetime import date
from pathlib import Path

import pandas as pd
import requests

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "_quarto.yml").exists())
RAW = ROOT / "data" / "raw"
PROC = ROOT / "data" / "processed"
TOPICS = RAW / "openalex_topics.csv"
SOURCES_RAW = RAW / "included_sources_raw.csv"
TOPIC_COLS = ["source_id", "record_id", "doi", "openalex_field", "openalex_subfield", "status", "pulled_on"]
EMAIL = "zachary.batist@mcgill.ca"


def primary_topic(doi):
    """Field and subfield of the primary topic of the work with this DOI in OpenAlex, and a status."""
    if not doi:
        return "", "", "no-doi"
    r = requests.get(f"https://api.openalex.org/works/doi:{doi}",
                     params={"select": "primary_topic", "mailto": EMAIL}, timeout=30)
    time.sleep(0.1)
    if r.status_code == 404:
        return "", "", "not-found"
    r.raise_for_status()
    topic = r.json().get("primary_topic")
    if not topic:
        return "", "", "no-topic"
    return topic["field"]["display_name"], topic["subfield"]["display_name"], "found"


inc_path, dedup_path = PROC / "included_reports.csv", PROC / "deduplicated_records.csv"
if not (inc_path.exists() and dedup_path.exists()):
    print("Not run yet. The OpenAlex pull needs included_reports.csv and deduplicated_records.csv.")
else:
    # One row per included source: its first report with a DOI, or its first report
    inc = pd.read_csv(inc_path, dtype=str).fillna("")
    dois = pd.read_csv(dedup_path, dtype=str).fillna("").set_index("record_id")["doi"]
    inc["doi"] = inc["record_id"].map(dois).fillna("")
    inc["has_doi"] = inc["doi"] != ""
    first = (inc.sort_values(["source_id", "has_doi", "record_id"], ascending=[True, False, True])
             .drop_duplicates("source_id"))

    # Pull the sources not pulled before; values already pulled never change
    pulled = pd.read_csv(TOPICS, dtype=str).fillna("") if TOPICS.exists() else pd.DataFrame(columns=TOPIC_COLS)
    new = first[~first["source_id"].isin(pulled["source_id"])]
    rows = []
    for s in new.itertuples():
        field, subfield, status = primary_topic(s.doi)
        rows.append(dict(source_id=s.source_id, record_id=s.record_id, doi=s.doi, openalex_field=field,
                         openalex_subfield=subfield, status=status, pulled_on=date.today().isoformat()))
    if rows:
        pulled = pd.concat([pulled, pd.DataFrame(rows, columns=TOPIC_COLS)], ignore_index=True)
        pulled.to_csv(TOPICS, index=False)
    print(f"Pulled {len(rows)} source(s) from OpenAlex; {len(pulled)} in openalex_topics.csv.")
    if rows:
        display(pd.DataFrame(rows)["status"].value_counts().rename("sources").reset_index())

    # Pre-fill a row in the charting file for each included source that has none; charted rows stay as they are
    text = SOURCES_RAW.read_text(encoding="utf-8")
    header = [h.strip().lower() for h in next(csv.reader([text.splitlines()[0]]))]
    charted = {r["source_id"].strip() for r in csv.DictReader(text.splitlines()[1:], fieldnames=header) if r["source_id"]}
    todo = pulled[pulled["source_id"].isin(first["source_id"]) & ~pulled["source_id"].isin(charted)]
    if len(todo):
        with open(SOURCES_RAW, "a", newline="", encoding="utf-8") as fh:
            if not text.endswith("\n"):
                fh.write("\n")
            w = csv.writer(fh, lineterminator="\n")
            for t in todo.itertuples():
                row = dict(source_id=t.source_id, record_id=t.record_id, openalex_field=t.openalex_field,
                           openalex_subfield=t.openalex_subfield,
                           openalex_corrected="" if t.status == "found" else "no-topic")
                w.writerow([row.get(h, "") for h in header])
    print(f"Pre-filled {len(todo)} row(s) in included_sources_raw.csv.")

In [2]:
if have_data:
    clean = {"sources": sources.copy(), "factors": factors.copy()}
    for name, df in clean.items():
        for v in TABLES[name]["variables"]:
            if v["values"]:
                df[v["name"]] = df[v["name"]].str.lower()

## Checks

In [ ]:
{
 "cells": [
  {
   "cell_type": "raw",
   "id": "97fae0ce",
   "metadata": {},
   "source": [
    "---\n",
    "title: \"03 Charting\"\n",
    "jupyter: python3\n",
    "execute:\n",
    "  echo: true\n",
    "  enabled: true\n",
    "format:\n",
    "  html:\n",
    "    code-fold: true\n",
    "    code-summary: \"Code\"\n",
    "---"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "51219efc",
   "metadata": {},
   "source": [
    "This notebook cleans the charted data and checks it against the [codebook](../codebook.qmd). The charting form has two tables. `data/raw/included_sources_raw.csv` has one row per included source. `data/raw/included_factors_raw.csv` has one row per factor that a source reports. The notebook writes `data/processed/included_sources_clean.csv`, `data/processed/included_factors_clean.csv`, and `data/processed/bibliography.csv`, which holds the reference details of each included source, taken from the deduplicated records.\n",
    "\n",
    "The variables, the closed value lists, and the required variables are read from `codebook.yml`, so this notebook holds no copy of them. Cleaning trims whitespace and lowercases the closed variables.\n",
    "\n",
    "The checks come first. If any check finds an error, the notebook stops and writes nothing. A warning is printed and does not stop the run. Errors are an empty required cell, a value outside the codebook, a repeated `source_id`, a factor whose `source_id` is not in the sources table, a `record_id` that is not in the deduplicated records, and a `source_id` that screening did not include. Warnings are sources with no factors, included sources not charted yet, repeated factor rows, and factors not yet coded to a category or level."
   ]
  },
  {
   "cell_type": "markdown",
   "id": "11397ada",
   "metadata": {},
   "source": [
    "## OpenAlex field and subfield\n",
    "\n",
    "This step takes the field and subfield of each included source's primary topic from OpenAlex by DOI, and saves them with the pull date to `data/raw/openalex_topics.csv`. It pulls only sources not pulled before, so values already pulled never change. It then adds a row to `included_sources_raw.csv` for each included source that has none, with the `source_id`, the `record_id` and the OpenAlex values, and `no-topic` in `openalex_corrected` where OpenAlex gives none. Rows already in the file stay as they are."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "b0af89ac",
   "metadata": {},
   "outputs": [],
   "source": [
    "import csv\n",
    "import time\n",
    "from datetime import date\n",
    "from pathlib import Path\n",
    "\n",
    "import pandas as pd\n",
    "import requests\n",
    "\n",
    "ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / \"_quarto.yml\").exists())\n",
    "RAW = ROOT / \"data\" / \"raw\"\n",
    "PROC = ROOT / \"data\" / \"processed\"\n",
    "TOPICS = RAW / \"openalex_topics.csv\"\n",
    "SOURCES_RAW = RAW / \"included_sources_raw.csv\"\n",
    "TOPIC_COLS = [\"source_id\", \"record_id\", \"doi\", \"openalex_field\", \"openalex_subfield\", \"status\", \"pulled_on\"]\n",
    "EMAIL = \"zachary.batist@mcgill.ca\"\n",
    "\n",
    "\n",
    "def primary_topic(doi):\n",
    "    \"\"\"Field and subfield of the primary topic of the work with this DOI in OpenAlex, and a status.\"\"\"\n",
    "    if not doi:\n",
    "        return \"\", \"\", \"no-doi\"\n",
    "    r = requests.get(f\"https://api.openalex.org/works/doi:{doi}\",\n",
    "                     params={\"select\": \"primary_topic\", \"mailto\": EMAIL}, timeout=30)\n",
    "    time.sleep(0.1)\n",
    "    if r.status_code == 404:\n",
    "        return \"\", \"\", \"not-found\"\n",
    "    r.raise_for_status()\n",
    "    topic = r.json().get(\"primary_topic\")\n",
    "    if not topic:\n",
    "        return \"\", \"\", \"no-topic\"\n",
    "    return topic[\"field\"][\"display_name\"], topic[\"subfield\"][\"display_name\"], \"found\"\n",
    "\n",
    "\n",
    "inc_path, dedup_path = PROC / \"included_reports.csv\", PROC / \"deduplicated_records.csv\"\n",
    "if not (inc_path.exists() and dedup_path.exists()):\n",
    "    print(\"Not run yet. The OpenAlex pull needs included_reports.csv and deduplicated_records.csv.\")\n",
    "else:\n",
    "    # One row per included source: its first report with a DOI, or its first report\n",
    "    inc = pd.read_csv(inc_path, dtype=str).fillna(\"\")\n",
    "    dois = pd.read_csv(dedup_path, dtype=str).fillna(\"\").set_index(\"record_id\")[\"doi\"]\n",
    "    inc[\"doi\"] = inc[\"record_id\"].map(dois).fillna(\"\")\n",
    "    inc[\"has_doi\"] = inc[\"doi\"] != \"\"\n",
    "    first = (inc.sort_values([\"source_id\", \"has_doi\", \"record_id\"], ascending=[True, False, True])\n",
    "             .drop_duplicates(\"source_id\"))\n",
    "\n",
    "    # Pull the sources not pulled before; values already pulled never change\n",
    "    pulled = pd.read_csv(TOPICS, dtype=str).fillna(\"\") if TOPICS.exists() else pd.DataFrame(columns=TOPIC_COLS)\n",
    "    new = first[~first[\"source_id\"].isin(pulled[\"source_id\"])]\n",
    "    rows = []\n",
    "    for s in new.itertuples():\n",
    "        field, subfield, status = primary_topic(s.doi)\n",
    "        rows.append(dict(source_id=s.source_id, record_id=s.record_id, doi=s.doi, openalex_field=field,\n",
    "                         openalex_subfield=subfield, status=status, pulled_on=date.today().isoformat()))\n",
    "    if rows:\n",
    "        pulled = pd.concat([pulled, pd.DataFrame(rows, columns=TOPIC_COLS)], ignore_index=True)\n",
    "        pulled.to_csv(TOPICS, index=False)\n",
    "    print(f\"Pulled {len(rows)} source(s) from OpenAlex; {len(pulled)} in openalex_topics.csv.\")\n",
    "    if rows:\n",
    "        display(pd.DataFrame(rows)[\"status\"].value_counts().rename(\"sources\").reset_index())\n",
    "\n",
    "    # Pre-fill a row in the charting file for each included source that has none; charted rows stay as they are\n",
    "    text = SOURCES_RAW.read_text(encoding=\"utf-8\")\n",
    "    header = [h.strip().lower() for h in next(csv.reader([text.splitlines()[0]]))]\n",
    "    charted = {r[\"source_id\"].strip() for r in csv.DictReader(text.splitlines()[1:], fieldnames=header) if r[\"source_id\"]}\n",
    "    todo = pulled[pulled[\"source_id\"].isin(first[\"source_id\"]) & ~pulled[\"source_id\"].isin(charted)]\n",
    "    if len(todo):\n",
    "        with open(SOURCES_RAW, \"a\", newline=\"\", encoding=\"utf-8\") as fh:\n",
    "            if not text.endswith(\"\\n\"):\n",
    "                fh.write(\"\\n\")\n",
    "            w = csv.writer(fh, lineterminator=\"\\n\")\n",
    "            for t in todo.itertuples():\n",
    "                row = dict(source_id=t.source_id, record_id=t.record_id, openalex_field=t.openalex_field,\n",
    "                           openalex_subfield=t.openalex_subfield,\n",
    "                           openalex_corrected=\"\" if t.status == \"found\" else \"no-topic\")\n",
    "                w.writerow([row.get(h, \"\") for h in header])\n",
    "    print(f\"Pre-filled {len(todo)} row(s) in included_sources_raw.csv.\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 1,
   "id": "941a8e07",
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "Nothing charted yet. Add rows to the two charting files in data/raw/.\n"
     ]
    }
   ],
   "source": [
    "from pathlib import Path\n",
    "\n",
    "import pandas as pd\n",
    "import yaml\n",
    "\n",
    "ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / \"_quarto.yml\").exists())\n",
    "PROC = ROOT / \"data\" / \"processed\"\n",
    "DEDUP = PROC / \"deduplicated_records.csv\"\n",
    "INCLUDED = PROC / \"included_reports.csv\"\n",
    "\n",
    "TABLES = yaml.safe_load((ROOT / \"codebook.yml\").read_text(encoding=\"utf-8\"))[\"tables\"]\n",
    "for t in TABLES.values():\n",
    "    for v in t[\"variables\"]:\n",
    "        v[\"values\"] = v.get(\"values\", [])\n",
    "        if not all(isinstance(x, str) for x in v[\"values\"]):\n",
    "            raise ValueError(f'codebook.yml: the values of {v[\"name\"]} must be text. Put quotes around values such as \"yes\", \"no\", and \"1\".')\n",
    "\n",
    "\n",
    "def load(name):\n",
    "    t = TABLES[name]\n",
    "    path = ROOT / t[\"file\"]\n",
    "    cols = [v[\"name\"] for v in t[\"variables\"]]\n",
    "    if not path.exists():\n",
    "        return pd.DataFrame(columns=cols)\n",
    "    df = pd.read_csv(path, dtype=str).fillna(\"\")\n",
    "    df.columns = [c.strip().lower() for c in df.columns]\n",
    "    absent = [c for c in cols if c not in df.columns]\n",
    "    if absent:\n",
    "        raise ValueError(f\"{path.name} is missing column(s): {', '.join(absent)}\")\n",
    "    return df[cols].apply(lambda s: s.str.strip())\n",
    "\n",
    "\n",
    "def sample(values, n=5):\n",
    "    values = sorted(set(map(str, values)))\n",
    "    return \", \".join(values[:n]) + (f\" and {len(values) - n} more\" if len(values) > n else \"\")\n",
    "\n",
    "\n",
    "def lines(mask):\n",
    "    # Line numbers in the CSV file: the header is line 1.\n",
    "    numbers = sorted(i + 2 for i in mask[mask].index)\n",
    "    return \", \".join(map(str, numbers[:5])) + (f\" and {len(numbers) - 5} more\" if len(numbers) > 5 else \"\")\n",
    "\n",
    "\n",
    "sources, factors = load(\"sources\"), load(\"factors\")\n",
    "have_data = len(sources) > 0 or len(factors) > 0\n",
    "print(f\"{len(sources)} charted source(s), {len(factors)} charted factor(s).\" if have_data else \"Nothing charted yet. Add rows to the two charting files in data/raw/.\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 2,
   "id": "809420ea",
   "metadata": {},
   "outputs": [],
   "source": [
    "if have_data:\n",
    "    clean = {\"sources\": sources.copy(), \"factors\": factors.copy()}\n",
    "    for name, df in clean.items():\n",
    "        for v in TABLES[name][\"variables\"]:\n",
    "            if v[\"values\"]:\n",
    "                df[v[\"name\"]] = df[v[\"name\"]].str.lower()"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "c25638fd",
   "metadata": {},
   "source": [
    "## Checks"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "31ab83f9",
   "metadata": {},
   "outputs": [],
   "source": [
    "if have_data:\n",
    "    errors, warnings = [], []\n",
    "\n",
    "    # Rows the OpenAlex pull pre-filled and nobody has charted yet: every required variable apart from\n",
    "    # source_id and record_id is empty. They leave the checks and the cleaned tables until they are charted.\n",
    "    required = [v[\"name\"] for v in TABLES[\"sources\"][\"variables\"]\n",
    "                if v.get(\"required\") and v[\"name\"] not in (\"source_id\", \"record_id\")]\n",
    "    uncharted = (clean[\"sources\"][required] == \"\").all(axis=1)\n",
    "    clean[\"sources\"] = clean[\"sources\"][~uncharted]\n",
    "    src, fac = clean[\"sources\"], clean[\"factors\"]\n",
    "\n",
    "    # Each table against the codebook\n",
    "    for name, df in clean.items():\n",
    "        file = Path(TABLES[name][\"file\"]).name\n",
    "        for v in TABLES[name][\"variables\"]:\n",
    "            col = df[v[\"name\"]]\n",
    "            if v.get(\"required\"):\n",
    "                empty = col == \"\"\n",
    "                if empty.any():\n",
    "                    errors.append(f\"{file}: {int(empty.sum())} row(s) have an empty {v['name']} (line {lines(empty)}).\")\n",
    "            if v[\"values\"]:\n",
    "                bad = (col != \"\") & ~col.isin(v[\"values\"])\n",
    "                if bad.any():\n",
    "                    found = sample(col[bad])\n",
    "                    errors.append(f\"{file}: {v['name']} has value(s) outside the codebook: {found} (line {lines(bad)}). Allowed: {', '.join(v['values'])}.\")\n",
    "\n",
    "    # Sources\n",
    "    repeated = src[\"source_id\"].duplicated(keep=False) & (src[\"source_id\"] != \"\")\n",
    "    if repeated.any():\n",
    "        errors.append(f\"included_sources_raw.csv: source_id values repeated: {sample(src.loc[repeated, 'source_id'])}.\")\n",
    "    if DEDUP.exists():\n",
    "        dedup_ids = set(pd.read_csv(DEDUP, dtype=str)[\"record_id\"])\n",
    "        unknown = set(src[\"record_id\"]) - dedup_ids - {\"\"}\n",
    "        if unknown:\n",
    "            errors.append(f\"included_sources_raw.csv: {len(unknown)} record_id value(s) are not in deduplicated_records.csv: {sample(unknown)}.\")\n",
    "    else:\n",
    "        warnings.append(\"deduplicated_records.csv not found, so record_id values were not checked.\")\n",
    "    if INCLUDED.exists():\n",
    "        included = set(pd.read_csv(INCLUDED, dtype=str)[\"source_id\"])\n",
    "        unknown = set(src[\"source_id\"]) - included - {\"\"}\n",
    "        if unknown:\n",
    "            errors.append(f\"included_sources_raw.csv: {len(unknown)} source_id value(s) were not included in screening: {sample(unknown)}.\")\n",
    "        waiting = included - set(src[\"source_id\"])\n",
    "        if waiting:\n",
    "            warnings.append(f\"{len(waiting)} included source(s) are not charted yet: {sample(waiting)}.\")\n",
    "    else:\n",
    "        warnings.append(\"included_reports.csv not found, so source_id values were not checked against screening.\")\n",
    "\n",
    "    # Factors\n",
    "    unknown = set(fac[\"source_id\"]) - set(src[\"source_id\"]) - {\"\"}\n",
    "    if unknown:\n",
    "        errors.append(f\"included_factors_raw.csv: {len(unknown)} source_id value(s) are not in included_sources_raw.csv: {sample(unknown)}.\")\n",
    "    without = set(src[\"source_id\"]) - set(fac[\"source_id\"]) - {\"\"}\n",
    "    if without:\n",
    "        warnings.append(f\"{len(without)} charted source(s) have no factor rows: {sample(without)}.\")\n",
    "    same = fac.duplicated([\"source_id\", \"factor\", \"factor_type\", \"level\"], keep=False)\n",
    "    if same.any():\n",
    "        warnings.append(f\"included_factors_raw.csv: {int(same.sum())} row(s) repeat another row (line {lines(same)}).\")\n",
    "    for col in (\"factor_category\", \"level\"):\n",
    "        n = int((fac[col] == \"\").sum())\n",
    "        if n:\n",
    "            warnings.append(f\"{n} factor row(s) have no {col} yet.\")\n",
    "\n",
    "    for w in warnings:\n",
    "        print(\"Warning:\", w)\n",
    "    if errors:\n",
    "        raise ValueError(\"Fix these problems in the charting files, then run this notebook again:\\n- \" + \"\\n- \".join(errors))\n",
    "    print(\"No errors found.\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 4,
   "id": "f6783f59",
   "metadata": {},
   "outputs": [],
   "source": [
    "if have_data:\n",
    "    src.to_csv(PROC / \"included_sources_clean.csv\", index=False)\n",
    "    fac.to_csv(PROC / \"included_factors_clean.csv\", index=False)\n",
    "    print(\"Wrote data/processed/included_sources_clean.csv and included_factors_clean.csv\")\n",
    "    if DEDUP.exists():\n",
    "        dedup = pd.read_csv(DEDUP, dtype=str).fillna(\"\")\n",
    "        keep = [\"record_id\", \"authors\", \"year\", \"title\", \"journal\", \"doi\"]\n",
    "        bib = src[[\"source_id\", \"record_id\"]].merge(dedup[keep], on=\"record_id\", how=\"left\")\n",
    "        bib.to_csv(PROC / \"bibliography.csv\", index=False)\n",
    "        print(\"Wrote data/processed/bibliography.csv\")\n",
    "        display(bib.head(20))\n",
    "    else:\n",
    "        print(\"deduplicated_records.csv not found, so bibliography.csv was not written.\")"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "c0f3a001",
   "metadata": {},
   "source": [
    "## Summary"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "c0f3a002",
   "metadata": {},
   "outputs": [],
   "source": [
    "if have_data:\n",
    "    display(src.groupby(\"field\").size().rename(\"sources\").reset_index())\n",
    "    if len(fac):\n",
    "        display(fac.groupby([\"factor_type\", \"factor_category\"], dropna=False).size().rename(\"factors\").reset_index())"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.14.6"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 5
}


In [4]:
if have_data:
    src.to_csv(PROC / "included_sources_clean.csv", index=False)
    fac.to_csv(PROC / "included_factors_clean.csv", index=False)
    print("Wrote data/processed/included_sources_clean.csv and included_factors_clean.csv")
    if DEDUP.exists():
        dedup = pd.read_csv(DEDUP, dtype=str).fillna("")
        keep = ["record_id", "authors", "year", "title", "journal", "doi"]
        bib = src[["source_id", "record_id"]].merge(dedup[keep], on="record_id", how="left")
        bib.to_csv(PROC / "bibliography.csv", index=False)
        print("Wrote data/processed/bibliography.csv")
        display(bib.head(20))
    else:
        print("deduplicated_records.csv not found, so bibliography.csv was not written.")

## Summary

In [ ]:
if have_data:
    display(src.groupby("field").size().rename("sources").reset_index())
    if len(fac):
        display(fac.groupby(["factor_type", "factor_category"], dropna=False).size().rename("factors").reset_index())